# Excel → Python → SQL → Power BI — Cheat Sheet Completo

[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vagnerx/excel-python-sql-powerbi-cheatsheet/blob/main/notebooks/cheatsheet_completo.ipynb)

**Repositório:** https://github.com/vagnerx/excel-python-sql-powerbi-cheatsheet  
**GitHub Pages:** https://vagnerx.github.io/excel-python-sql-powerbi-cheatsheet/

---

## O que você vai encontrar aqui

Este notebook contém todas as **24 operações de dados** do cheat sheet,
executadas em Python/Pandas com datasets reais.

| Grupo | Operações |
|---|---|
| **A — Manipulação Básica** | Importar, Filtrar, Selecionar, Ordenar, Agrupar, Contar, Média, Soma |
| **B — Transformação** | Valores Únicos, Renomear, Tipos, Nulos, Merge/Join, Coluna Condicional |
| **C — Datas e Métricas** | Min/Max, Datas, Filtrar por Data, Contar Únicos, Top N, Percentual |
| **D — Analytics Avançado** | Ranking, Acumulado, Janela Móvel, Pivot/Unpivot |

### Como usar no Google Colab
1. Clique no badge **Abrir no Google Colab** acima
2. Execute a célula de setup (primeira célula de código)
3. Execute as demais células em sequência

### Como usar localmente
```bash
pip install pandas numpy
jupyter notebook notebooks/cheatsheet_completo.ipynb
```

In [ ]:
# ─── SETUP — Execute esta célula primeiro ────────────────────────────────────
import os, sys
import pandas as pd
import numpy as np

# Detecta se está no Colab e clona o repositório se necessário
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone https://github.com/vagnerx/excel-python-sql-powerbi-cheatsheet.git
    DATASETS = 'excel-python-sql-powerbi-cheatsheet/datasets'
else:
    # Execução local — ajusta path para a raiz do projeto
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if os.path.basename(os.getcwd()) == 'notebooks':
        REPO_ROOT = os.path.abspath('..')
    else:
        REPO_ROOT = os.getcwd()
    DATASETS = os.path.join(REPO_ROOT, 'datasets')

# Carrega os datasets principais
emp   = pd.read_csv(os.path.join(DATASETS, 'employees.csv'))
cust  = pd.read_csv(os.path.join(DATASETS, 'customers.csv'))
ord_  = pd.read_csv(os.path.join(DATASETS, 'orders.csv'))

# Converte datas
emp['data_admissao'] = pd.to_datetime(emp['data_admissao'], errors='coerce')
ord_['data_pedido']  = pd.to_datetime(ord_['data_pedido'],  errors='coerce')

print(f'employees : {emp.shape}')
print(f'customers : {cust.shape}')
print(f'orders    : {ord_.shape}')
print('\nSetup OK!')

---
## GRUPO A — Manipulação Básica

In [ ]:
# ── 1. IMPORTAR DADOS ────────────────────────────────────────────────────────
print('=== 1. IMPORTAR DADOS ===')
print(f'Shape: {emp.shape}')
print(f'Colunas: {list(emp.columns)}')
emp.head()

In [ ]:
# Informações do dataset
emp.info()

In [ ]:
# ── 2. FILTRAR LINHAS ────────────────────────────────────────────────────────
print('=== 2. FILTRAR LINHAS ===')

# Filtro simples
senior = emp[emp['salario'] > 8000]
print(f'Salário > 8000: {len(senior)} funcionários')

# Filtro composto (E)
ti_senior = emp[(emp['departamento'] == 'TI') & (emp['salario'] > 7000)]
print(f'TI E salário > 7000: {len(ti_senior)}')

# Filtro com lista
deptos = ['TI', 'Financeiro']
selecionados = emp[emp['departamento'].isin(deptos)]
print(f'TI ou Financeiro: {len(selecionados)}')

# query() — sintaxe mais legível
resultado = emp.query('salario > 7000 and departamento == "Vendas"')
print(f'Vendas salário > 7000: {len(resultado)}')

senior[['nome', 'departamento', 'salario']].head()

In [ ]:
# ── 3. SELECIONAR COLUNAS ────────────────────────────────────────────────────
print('=== 3. SELECIONAR COLUNAS ===')

# Selecionar colunas específicas
resumo = emp[['nome', 'departamento', 'salario']]

# Excluir coluna
sem_id = emp.drop(columns=['id'])

# Apenas colunas numéricas
numericas = emp.select_dtypes(include=['number'])

print(f'Colunas selecionadas: {list(resumo.columns)}')
print(f'Sem id: {list(sem_id.columns)}')
print(f'Só numéricas: {list(numericas.columns)}')
resumo.head()

In [ ]:
# ── 4. ORDENAR DADOS ─────────────────────────────────────────────────────────
print('=== 4. ORDENAR DADOS ===')

# Ordem decrescente
por_salario = emp.sort_values('salario', ascending=False)

# Multi-coluna
multi = emp.sort_values(['departamento', 'salario'], ascending=[True, False])

# Top 5 maiores
top5 = emp.nlargest(5, 'salario')

print('Top 5 maiores salários:')
top5[['nome', 'departamento', 'salario']]

In [ ]:
# ── 5. AGRUPAR / AGREGAR ─────────────────────────────────────────────────────
print('=== 5. AGRUPAR / AGREGAR ===')

resumo = emp.groupby('departamento').agg(
    funcionarios   = ('id',      'count'),
    salario_medio  = ('salario', 'mean'),
    salario_total  = ('salario', 'sum'),
    maior_salario  = ('salario', 'max'),
).round(2)

resumo.sort_values('salario_total', ascending=False)

In [ ]:
# ── 6. CONTAR LINHAS ─────────────────────────────────────────────────────────
print('=== 6. CONTAR LINHAS ===')
print(f'Total de linhas: {len(emp)}')
print(f'Funcionários ativos: {(emp["ativo"] == "Sim").sum()}')
print(f'Nulos em salário: {emp["salario"].isna().sum()}')
print()
emp.groupby('departamento')['id'].count().rename('qtd').sort_values(ascending=False)

In [ ]:
# ── 7. MÉDIA ─────────────────────────────────────────────────────────────────
print('=== 7. MÉDIA ===')
print(f'Média geral: R$ {emp["salario"].mean():,.2f}')
print(f'Mediana:     R$ {emp["salario"].median():,.2f}')

emp.groupby('departamento')['salario'].mean().round(2).sort_values(ascending=False)

In [ ]:
# ── 8. SOMA ──────────────────────────────────────────────────────────────────
print('=== 8. SOMA ===')
total = emp['salario'].sum()
print(f'Folha total: R$ {total:,.2f}')

# Soma por depto + percentual
por_depto = emp.groupby('departamento')['salario'].sum()
pct = (por_depto / total * 100).round(1)
resultado = pd.DataFrame({'total': por_depto, 'pct_%': pct}).sort_values('total', ascending=False)
resultado

---
## GRUPO B — Transformação

In [ ]:
# ── 9. OBTER VALORES ÚNICOS ──────────────────────────────────────────────────
print('=== 9. OBTER VALORES ÚNICOS ===')
print(f'Departamentos únicos ({emp["departamento"].nunique()}):', emp['departamento'].unique())

# Detectar duplicatas
dups = emp[emp.duplicated()]
print(f'\nLinhas duplicadas: {len(dups)}')

# Frequência
emp['departamento'].value_counts()

In [ ]:
# ── 10. RENOMEAR COLUNAS ─────────────────────────────────────────────────────
print('=== 10. RENOMEAR COLUNAS ===')
df_renamed = emp.rename(columns={
    'nome':         'funcionario',
    'departamento': 'depto',
    'salario':      'salario_bruto',
})
print('Novas colunas:', list(df_renamed.columns))
df_renamed.head(3)

In [ ]:
# ── 11. TIPOS DE DADOS ───────────────────────────────────────────────────────
print('=== 11. TIPOS DE DADOS ===')
print('Tipos originais:')
print(emp.dtypes)

df_typed = emp.copy()
df_typed['salario']  = pd.to_numeric(df_typed['salario'], errors='coerce')
df_typed['departamento'] = df_typed['departamento'].astype('category')

print('\nApós conversões:')
print(df_typed.dtypes)

In [ ]:
# ── 12. TRATAR NULOS ─────────────────────────────────────────────────────────
print('=== 12. TRATAR NULOS ===')
print('Nulos por coluna:')
print(emp.isna().sum())

df_limpo = emp.copy()
media_sal = df_limpo['salario'].mean()
df_limpo['salario']      = df_limpo['salario'].fillna(media_sal)
df_limpo['departamento'] = df_limpo['departamento'].fillna('Não informado')
df_limpo['cargo']        = df_limpo['cargo'].fillna('Não informado')

print('\nApós tratamento:')
print(df_limpo.isna().sum())

In [ ]:
# ── 13. MERGE / JOIN ─────────────────────────────────────────────────────────
print('=== 13. MERGE / JOIN ===')

# INNER JOIN: orders + customers
inner = pd.merge(ord_, cust, on='id_cliente', how='inner')
print(f'INNER JOIN: {len(inner)} linhas')

# LEFT JOIN
left = pd.merge(ord_, cust, on='id_cliente', how='left')
print(f'LEFT JOIN:  {len(left)} linhas')

# Agrupado por cliente
por_cliente = (
    inner.groupby(['id_cliente', 'nome'])
    .agg(qtd_pedidos=('id_pedido','count'), total_gasto=('valor_total','sum'))
    .round(2)
    .sort_values('total_gasto', ascending=False)
)
por_cliente.head()

In [ ]:
# ── 14. COLUNA CONDICIONAL ───────────────────────────────────────────────────
print('=== 14. COLUNA CONDICIONAL ===')

df = emp.copy()

# np.select — múltiplas condições
conds  = [df['salario'] > 12000, df['salario'] > 8000, df['salario'] > 5000]
niveis = ['Especialista',        'Sênior',              'Pleno']
df['nivel'] = np.select(conds, niveis, default='Júnior')

print('Distribuição de níveis:')
print(df['nivel'].value_counts())
df[['nome', 'salario', 'nivel']].head(8)

---
## GRUPO C — Datas e Métricas

In [ ]:
# ── 15. MIN / MAX ────────────────────────────────────────────────────────────
print('=== 15. MIN / MAX ===')
df = emp.dropna(subset=['salario'])

print(f'Menor salário: R$ {df["salario"].min():,.2f}')
print(f'Maior salário: R$ {df["salario"].max():,.2f}')
print(f'Amplitude:     R$ {df["salario"].max() - df["salario"].min():,.2f}')

# Quem tem o maior salário
max_row = df.loc[df['salario'].idxmax()]
print(f'\nMaior salário: {max_row["nome"]} — {max_row["departamento"]} — R$ {max_row["salario"]:,.2f}')

# Min/Max por depto
df.groupby('departamento')['salario'].agg(['min','max']).round(2)

In [ ]:
# ── 16. EXTRAÇÃO DE DATAS ────────────────────────────────────────────────────
print('=== 16. EXTRAÇÃO DE DATAS ===')
df = emp.dropna(subset=['data_admissao']).copy()

df['ano']       = df['data_admissao'].dt.year
df['mes']       = df['data_admissao'].dt.month
df['trimestre'] = df['data_admissao'].dt.quarter
df['nome_mes']  = df['data_admissao'].dt.month_name()

# Tempo de casa
df['anos_empresa'] = ((pd.Timestamp.today() - df['data_admissao']).dt.days / 365.25).round(1)

print('Por ano:')
print(df.groupby('ano')['id'].count().rename('contratacoes'))

df[['nome','data_admissao','ano','trimestre','anos_empresa']].head()

In [ ]:
# ── 17. FILTRAR POR DATA ─────────────────────────────────────────────────────
print('=== 17. FILTRAR POR DATA ===')
df = emp.dropna(subset=['data_admissao'])

em_2023   = df[df['data_admissao'].dt.year == 2023]
intervalo = df[df['data_admissao'].between('2022-01-01', '2023-12-31')]

print(f'Admitidos em 2023: {len(em_2023)}')
print(f'Admitidos 2022-2023: {len(intervalo)}')

# Pedidos por mês
ord_.groupby(ord_['data_pedido'].dt.to_period('M'))['valor_total'].sum().tail(10)

In [ ]:
# ── 18. CONTAR VALORES ÚNICOS ────────────────────────────────────────────────
print('=== 18. CONTAR VALORES ÚNICOS ===')
print(f'Departamentos únicos: {emp["departamento"].nunique()}')
print(f'Cargos únicos:        {emp["cargo"].nunique()}')
print(f'Clientes com pedidos: {ord_["id_cliente"].nunique()}')

# Por grupo
emp.groupby('departamento')['cargo'].nunique().rename('cargos_unicos')

In [ ]:
# ── 19. TOP N REGISTROS ──────────────────────────────────────────────────────
print('=== 19. TOP N ===')

# Top 5 global
top5 = emp.nlargest(5, 'salario')[['nome','departamento','salario']]

# Top 2 por departamento
top2_depto = (
    emp.dropna(subset=['salario'])
    .sort_values('salario', ascending=False)
    .groupby('departamento')
    .head(2)
    .sort_values(['departamento','salario'], ascending=[True,False])
)

print('Top 5 salários:')
display(top5)
print('Top 2 por departamento:')
top2_depto[['departamento','nome','salario']]

In [ ]:
# ── 20. PERCENTUAL ──────────────────────────────────────────────────────────
print('=== 20. PERCENTUAL ===')
df = emp.dropna(subset=['salario']).copy()
total = df['salario'].sum()

# % por departamento
pct = (
    df.groupby('departamento')['salario'].sum()
    .div(total).mul(100).round(1)
    .rename('pct_folha_%')
    .sort_values(ascending=False)
)

# % dentro do departamento
df['total_depto'] = df.groupby('departamento')['salario'].transform('sum')
df['pct_depto']   = (df['salario'] / df['total_depto'] * 100).round(1)

print('Participação % na folha por depto:')
display(pct)
df[['nome','departamento','salario','pct_depto']].head(8)

---
## GRUPO D — Analytics Avançado

In [ ]:
# ── 21. RANKING ──────────────────────────────────────────────────────────────
print('=== 21. RANKING ===')
df = emp.dropna(subset=['salario']).copy()

df['rank_geral'] = df['salario'].rank(method='dense', ascending=False).astype('Int64')
df['rank_depto'] = (
    df.groupby('departamento')['salario']
    .rank(method='dense', ascending=False)
    .astype('Int64')
)

print('Top 10 por ranking geral:')
df.sort_values('rank_geral')[['rank_geral','nome','departamento','salario','rank_depto']].head(10)

In [ ]:
# ── 22. ACUMULADO ────────────────────────────────────────────────────────────
print('=== 22. ACUMULADO ===')
mensal = (
    ord_.dropna(subset=['data_pedido'])
    .groupby(ord_['data_pedido'].dt.to_period('M'))['valor_total']
    .sum()
    .reset_index()
)
mensal.columns = ['mes', 'receita_mes']
mensal['acumulado'] = mensal['receita_mes'].cumsum().round(2)
mensal['crescimento_pct'] = (mensal['receita_mes'].pct_change() * 100).round(1)
mensal.tail(12)

In [ ]:
# ── 23. JANELA MÓVEL ────────────────────────────────────────────────────────
print('=== 23. JANELA MÓVEL ===')
diario = (
    ord_.dropna(subset=['data_pedido'])
    .groupby('data_pedido')['valor_total'].sum()
    .reset_index()
    .sort_values('data_pedido')
    .set_index('data_pedido')
)
diario['mm_7d']  = diario['valor_total'].rolling('7D').mean().round(2)
diario['mm_14d'] = diario['valor_total'].rolling('14D').mean().round(2)
diario.tail(15)

In [ ]:
# ── 24. PIVOT / UNPIVOT ──────────────────────────────────────────────────────
print('=== 24. PIVOT / UNPIVOT ===')

# PIVOT
pivot = emp.dropna(subset=['salario','departamento','ativo']).pivot_table(
    values='salario',
    index='departamento',
    columns='ativo',
    aggfunc='mean',
    fill_value=0,
).round(2)

print('PIVOT — Média salarial por depto e status:')
display(pivot)

# UNPIVOT (melt)
long = ord_.melt(
    id_vars=['id_pedido','id_produto'],
    value_vars=['valor_unit','valor_total'],
    var_name='tipo',
    value_name='valor',
)
print(f'\nUNPIVOT — {len(long)} linhas')
long.head()

---
## Próximos passos

- Veja os exemplos de **SQL** no notebook [`sql_com_python.ipynb`](sql_com_python.ipynb)
- Explore a documentação completa no **GitHub Pages**: https://vagnerx.github.io/excel-python-sql-powerbi-cheatsheet/
- Download de todos os arquivos: [Repositório GitHub](https://github.com/vagnerx/excel-python-sql-powerbi-cheatsheet)

---
*Feito por Vagner Xavier — compartilhe e deixe uma ⭐ no repositório!*